# VoiceGuard — Kaggle training notebook

Run tonight on a Kaggle GPU runtime. Steps:
1. Clone the repo (or upload it as a Kaggle dataset/utility script) so `config.py` and `src/` are importable.
2. Attach the ASVspoof 2019 LA (+ 2021 DF / In-the-Wild) datasets via **Add Data** in the Kaggle sidebar — zero download to disk.
3. Train, evaluate EER on the unseen set, generate demo clips, save everything under `artifacts/` and `demo_clips/`.
4. Download `artifacts/` + `demo_clips/` and upload to Google Drive (see Step 10 of the runbook).

In [ ]:
# --- 1. Get the code onto the Kaggle runtime ---
!git clone https://github.com/BuiltByPriyanshu/Voiceguard.git /kaggle/working/voiceguard
%cd /kaggle/working/voiceguard

In [ ]:
# --- 2. Install deps ---
!pip install -q -r requirements_cuda.txt

In [ ]:
# --- 3. Confirm GPU + device selection ---
import sys; sys.path.insert(0, '/kaggle/working/voiceguard')
from src.device import get_device
print('device:', get_device())
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- 4. Locate attached datasets ---
# After using 'Add Data' in the Kaggle sidebar to attach ASVspoof 2019 LA
# (and optionally 2021 DF / In-the-Wild), list what actually landed under
# /kaggle/input so we can point --protocol / --audio-dir at the right paths.
!ls /kaggle/input
!find /kaggle/input -maxdepth 3 -iname '*.txt' | head -20
!find /kaggle/input -maxdepth 3 -type d | head -40

In [ ]:
# --- 5. Train (Step 5) -- FAST PATH on precomputed wav2vec2 embeddings ---
# Uses eminkorkut/deepfakevoice-wac2vec-4datasets: 768-dim wav2vec2 embeddings
# (matches facebook/wav2vec2-base, set in config.SSL_MODEL_NAME) already
# extracted from 2s segments, so this trains in minutes -- no raw audio, no
# SSL forward pass. Make sure this dataset is attached via Add Data first.
WAC2VEC_DIR = '/kaggle/input/datasets/eminkorkut/deepfakevoice-wac2vec-4datasets'
TRAIN_PARQUET = f'{WAC2VEC_DIR}/ASVspoof2019_train_wav2vec.parquet'

!python -m src.train_embeddings --parquet "$TRAIN_PARQUET" --epochs 15

In [ ]:
# --- 6. Cross-dataset EER on an UNSEEN set (Step 6) -- fast path ---
# ASVspoof2021 DF is the unseen-attack test set; In-the-wild is real-world
# generalisation. Both come precomputed from the same wac2vec dataset.
UNSEEN_PARQUET = f'{WAC2VEC_DIR}/ASVspoof2021_test_wav2vec.parquet'

!python -m src.eval_eer_embeddings \
  --parquet "$UNSEEN_PARQUET" \
  --dataset-name 'ASVspoof2021-DF'

# Also report on In-the-wild for the "real-world generalisation" story:
INTHEWILD_PARQUET = f'{WAC2VEC_DIR}/In-the-wild_test_wav2vec.parquet'
!python -m src.eval_eer_embeddings \
  --parquet "$INTHEWILD_PARQUET" \
  --dataset-name 'In-the-Wild'

!cat artifacts/metrics.json

## 7. Streaming inference sanity check -- deferred to Step 8b

The wac2vec dataset only ships precomputed embeddings, not raw audio, so
there's nothing to run `src.infer` against yet. The first real end-to-end
check of the `RiskEngine` (raw audio -> wav2vec2-base -> trained head -> risk
score) happens in **Step 8b** below, once we've generated real genuine +
cloned WAV clips.

## 8. Generate the demo clip pack (Step 8)
Upload a 10-20s clean reference sample of a teammate as `teammate_ref.wav`
into `/kaggle/working/voiceguard/demo_clips/` (via Kaggle's file upload) before
running this cell.

In [ ]:
# Coqui's original `TTS` PyPI package caps out at Python <3.12 and fails to
# resolve on newer Kaggle runtimes -- use the actively-maintained fork
# (`coqui-tts`), which keeps the same `TTS` import name.
!pip install -q coqui-tts

In [ ]:
import os
os.makedirs('demo_clips', exist_ok=True)

from TTS.api import TTS
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')

REF = 'demo_clips/teammate_ref.wav'

tts.tts_to_file(
    text='Hi, please transfer two lakh rupees to this account urgently.',
    speaker_wav=REF, language='en', file_path='demo_clips/fraud_en.wav')

tts.tts_to_file(
    text='\u0928\u092e\u0938\u094d\u0924\u0947, \u0915\u0943\u092a\u092f\u093e \u0907\u0938 \u0916\u093e\u0924\u0947 \u092e\u0947\u0902 \u0924\u0941\u0930\u0902\u0924 \u0926\u094b \u0932\u093e\u0916 \u0930\u0941\u092a\u092f\u0947 \u092d\u0947\u091c\u093f\u090f\u0964',
    speaker_wav=REF, language='hi', file_path='demo_clips/fraud_hi.wav')

print('Generated:', os.listdir('demo_clips'))

In [ ]:
# --- 8b. Sanity check the generated clips through the RiskEngine ---
# Genuine (REF) should score low; cloned (fraud_en/fraud_hi) should score high.
!python -m src.infer demo_clips/teammate_ref.wav
!python -m src.infer demo_clips/fraud_en.wav
!python -m src.infer demo_clips/fraud_hi.wav

## 9. Package the handoff bundle (Step 10)
Zip `artifacts/` and `demo_clips/`, download from Kaggle, upload to Google Drive.
Also push the code (already cloned from GitHub) if you made local edits in this notebook session.

In [ ]:
!zip -r voiceguard_handoff.zip artifacts demo_clips
print('Ready to download voiceguard_handoff.zip and upload to Google Drive.')